# 02. Exploratory Data Analysis & Feature Engineering
## AI-Based Technical Skill Development Assistant - Phase 1

**Author:** Member 2 (EDA, Visual Analytics, Composite Indicators & Leakage-Free Splitting)  
**Objective:** Comprehensive exploratory data analysis of technical competencies, correlation profiling, domain-specific composite feature engineering, and leakage-safe preprocessing.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.data_loader import SKILL_COLUMNS, load_raw_data
from src.feature_engineering import SkillFeaturePipeline
from src.preprocess import split_data_leak_free

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

df = load_raw_data(Path("../data/raw/skills_dataset.csv"))
print(f"Loaded {df.shape[0]} candidate records with {df.shape[1]} attributes.")
df[SKILL_COLUMNS].describe().T

### 1. Technical Competency Correlation Heatmap
Examines collinearity and skill domain clusters across the 17 technical disciplines.

In [ ]:
plt.figure(figsize=(14, 10))
corr = df[SKILL_COLUMNS].corr()
sns.heatmap(corr, cmap="coolwarm", annot=False, vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix of 17 Technical Skill Competencies", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

### 2. Role-Specific Technical Proficiency Profiles
Displays mean benchmark proficiency for the top career tracks.

In [ ]:
top_roles = df["target_role"].value_counts().head(8).index
role_profiles = df[df["target_role"].isin(top_roles)].groupby("target_role")[SKILL_COLUMNS].mean().T

plt.figure(figsize=(14, 8))
sns.heatmap(role_profiles, cmap="YlGnBu", annot=True, fmt=".1f", linewidths=0.5)
plt.title("Mean Technical Proficiency by Career Role (0-6 Scale)", fontsize=14, weight="bold")
plt.xlabel("Target Career Role")
plt.ylabel("Technical Competency")
plt.tight_layout()
plt.show()

### 3. Leakage-Free Feature Engineering & Scaling Pipeline
Engineers composite readiness indices:
- `ai_data_readiness`
- `systems_readiness`
- `software_dev_readiness`
- `cybersecurity_readiness`
- `business_comm_readiness`
- `analytical_mindset_index`

Applies standard scaling fit strictly on the training partition.

In [ ]:
train_df, test_df = split_data_leak_free(df, test_size=0.2, random_state=42)

pipeline = SkillFeaturePipeline()
train_proc, y_train = pipeline.fit_transform(train_df)
test_proc, y_test = pipeline.transform(test_df)

print(f"Transformed Training Partition: {train_proc.shape}")
print(f"Transformed Test Partition:     {test_proc.shape}")
print(f"Total Engineered Features:      {len(pipeline.feature_columns)}")
print(f"Target Career Classes:          {len(pipeline.class_mapping)}")
train_proc.head()

### 4. Member 2 Analytical Takeaways
1. **Leakage-Safe Partitioning:** Splitting precedes scaling and composite calculations, guaranteeing zero evaluation data leakage.
2. **Domain Signatures:** High-impact skills (e.g. AI/ML, Cyber Security, Software Development) exhibit distinct discriminative regions.
3. **Engineered Readiness Features:** The 5 composite indicators synthesize multi-skill aptitude into actionable domain scores for downstream modeling.